# 06 阈值反推（任务书：预测BCF + 国标限量 → 土壤Cd阈值）

In [2]:
import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
df = pd.read_parquet(OUT/'bcf_final.parquet')


In [3]:
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.thresholds import infer_soil_cd_thresholds
from src.utils import safe_col
import pandas as pd

bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
cv = get_group_kfold(cfg)

preferred = [m for m in ['xgb','rf'] if m in cfg['modeling']['models']]
if not preferred:
    raise RuntimeError("请在config里至少包含 xgb 或 rf 以高效做阈值反推")
m = preferred[0]

if 'crop' not in df.columns:
    raise RuntimeError("阈值反推需要crop列。请使用1.27文件(sheets模式)或确保crop可解析。")

n_rows = int(cfg['thresholds']['n_rows_per_group'])
all_thr = []
for (crop, phbin), sub in df.groupby(['crop','ph_bin'], dropna=False):
    sub = sub.dropna(subset=[bcf_col]).reset_index(drop=True)
    if len(sub) == 0: 
        continue
    sub_run = sub.sample(n_rows, random_state=cfg['project']['seed']).reset_index(drop=True) if len(sub)>n_rows else sub

    gsub = make_spatial_groups(sub_run, cfg)
    res = fit_oof_with_spatialcv(sub_run, cfg, gsub, m, cv)

    crop_key = 'potato' if crop == 'potato' else 'corn'
    thr = infer_soil_cd_thresholds(sub_run, cfg, res.best_estimator, crop_key=crop_key)
    thr = thr[['crop','ph_bin']].assign(model_used=m, soil_cd_threshold_mgkg=thr['soil_cd_threshold_mgkg'],
                                       crop_cd_limit_mgkg=thr['crop_cd_limit_mgkg'], gb_standard=thr['gb_standard'])
    all_thr.append(thr)

out = pd.concat(all_thr, ignore_index=True)
out.to_excel(OUT/'thresholds_by_ph.xlsx', index=False)

summary = out.groupby(['crop','ph_bin'])['soil_cd_threshold_mgkg'].quantile([0.05,0.25,0.5,0.75,0.95]).unstack()
summary.to_excel(OUT/'thresholds_summary_quantiles.xlsx')
summary


0.05      0.25      0.50      0.75      0.95
crop   ph_bin                                                       
corn   acid         0.221278  0.784200  5.403615  6.505102  9.006674
       alkaline     0.180887  0.649980  2.051893  4.206973  6.362791
       neutral      0.565183  0.723588  3.796064  4.467776  8.867960
       strong_acid  0.346533  0.590709  1.148190  6.916110  8.057813
potato acid         0.433844  0.557582  5.135436  9.124691  9.837215
       alkaline     0.685772  0.847984  4.434031  8.216867  9.700135
       neutral      0.384188  0.682190  0.726535  1.175335  7.552337
       strong_acid  0.240908  0.284616  3.799962  8.988087  9.688003